# Module 06 — ANOVA một chiều, hậu kiểm Tukey, ANOVA hai chiều
Notebook RIÊNG cho Module 06 — tái lập ANOVA một chiều + Tukey HSD (PSPP làm được, xem `06_anova.sps`), CỘNG THÊM ANOVA hai chiều + hiệu ứng tương tác (PSPP 2.1.1 CHƯA hỗ trợ `UNIANOVA` — chỉ làm được ở đây bằng `statsmodels`).

Tương ứng SPSS: `Analyze > Compare Means > One-Way ANOVA` (+ Post Hoc Tukey), `Analyze > General Linear Model > Univariate` (hai chiều).

In [ ]:
# Chạy ô này đầu tiên (Colab: bấm ▶). Không cần cài gì thêm.
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
sns.set_theme(style="whitegrid"); plt.rcParams["figure.figsize"] = (7, 4)
pd.set_option("display.precision", 3)

def make_data(seed=2026, n=240):
    """Bộ dữ liệu GIẢ LẬP 'Lớp học 240 học sinh' — CÙNG một seed=2026 dùng
    xuyên suốt các module 03-08, khớp chính xác với lop_hoc_240.csv/.sav
    trong repo (không phải dữ liệu thật, nhưng tái lập được 100%)."""
    rng = np.random.default_rng(seed)
    school = rng.choice(list("ABC"), n, p=[.35, .35, .30])
    gender = rng.choice(["Nam", "Nữ"], n)
    method = rng.choice(["Truyền thống", "Dự án"], n)
    study_hours = np.clip(rng.gamma(4, 1.2, n), 0.5, 15).round(1)
    interest = rng.normal(0, 1, n) + 0.15 * (method == "Dự án")
    anxiety = rng.normal(0, 1, n) - 0.25 * interest
    def likert(lat, load, noise=0.7):
        return np.clip(np.round(3 + load * lat + rng.normal(0, noise, n)), 1, 5).astype(int)
    d = pd.DataFrame({"id": np.arange(1, n + 1), "school": school, "gender": gender, "method": method,
                      "study_hours": study_hours})
    for k, (lat, load) in enumerate([(interest, .8), (interest, .7), (interest, .75)], 1):
        d[f"h{k}"] = likert(lat, load)
    for k, (lat, load) in enumerate([(anxiety, .8), (anxiety, .75), (anxiety, .7)], 1):
        d[f"a{k}"] = likert(lat, load)
    eff = d.school.map({"A": 3, "B": 0, "C": -3}).to_numpy()
    d["pretest"] = (rng.normal(60, 10, n) + eff).round(1)
    d["posttest"] = np.clip(d.pretest + 3 + 5 * (method == "Dự án") + rng.normal(0, 6, n), 0, 100).round(1)
    d["math"] = np.clip(35 + 2.5 * study_hours + 4 * interest - 3 * anxiety + eff + rng.normal(0, 6, n), 0, 100).round(1)
    d.loc[6, "study_hours"] = 48.0
    d.loc[[11, 57, 130], "h2"] = np.nan
    d["h2"] = d["h2"].astype("Int64")
    return d

df = make_data()
print("Đã tạo df:", df.shape)

## 1. ANOVA một chiều: điểm Toán theo 3 trường

🎯 **Phương pháp này trả lời câu hỏi gì?** ANOVA một chiều kiểm tra: trung bình của 3+ nhóm có ÍT NHẤT MỘT cặp khác nhau có ý nghĩa thống kê hay không — giả thuyết không (H0): TẤT CẢ các nhóm có cùng trung bình tổng thể. Quan trọng: ANOVA có ý nghĩa chỉ nói “có khác biệt Ở ĐÂU ĐÓ”, KHÔNG cho biết cặp nào khác nhau — đó là việc của hậu kiểm Tukey (mục 2 bên dưới).

In [ ]:
groups = [g["math"].values for _, g in df.groupby("school")]
f_stat, p_val = stats.f_oneway(*groups)
lev_stat, lev_p = stats.levene(*groups)
print(f"Levene: F={lev_stat:.3f}, p={lev_p:.3f}")
for name, g in df.groupby("school"):
    print(f"  Trường {name}: n={len(g)}, M={g['math'].mean():.2f}")
print(f"ANOVA: F(2,{len(df)-3})={f_stat:.3f}, p={p_val:.4f}")

#### 📤 Đầu ra thật
Trường A (n=80) TB≈50,67; B (n=99) TB≈47,26; C (n=61) TB≈45,34. F(2,237)=5,87, p=0,003 → có khác biệt có ý nghĩa. Levene p=0,292 (≥0,05, phương sai đồng nhất, đủ điều kiện dùng Tukey).

## 2. Hậu kiểm Tukey HSD — tìm CHÍNH XÁC cặp nào khác biệt

🎯 **Phương pháp này trả lời câu hỏi gì?** Tukey HSD kiểm tra TỪNG CẶP nhóm một cách riêng biệt, có điều chỉnh để không bị “thổi phồng” tỉ lệ sai lầm loại I do so sánh nhiều cặp cùng lúc (multiple comparisons problem — càng so sánh nhiều cặp, càng dễ tình cờ tìm ra 1 cặp “có ý nghĩa” dù thực ra không có). Đây chính là câu hỏi mà ANOVA ở mục 1 để lại chưa trả lời: khác biệt CHÍNH XÁC ở đâu.

In [ ]:
from statsmodels.stats.multicomp import pairwise_tukeyhsd
tukey = pairwise_tukeyhsd(df["math"], df["school"], alpha=0.05)
print(tukey)

#### 📤 Đầu ra thật
A–B chênh 3,41 (p≈0,046, có ý nghĩa); A–C chênh 5,34 (p≈0,003, có ý nghĩa); B–C chênh 1,93 (p≈0,427, không có ý nghĩa). Nhóm đồng nhất: {A} tách riêng, {B, C} cùng một nhóm.

## 3. ANOVA hai chiều: điểm Toán ~ trường × giới tính (+ hiệu ứng tương tác)
⚠️ PSPP 2.1.1 KHÔNG hỗ trợ lệnh `UNIANOVA` (đã kiểm tra, báo lỗi "UNIANOVA is not yet implemented") — phần này CHỈ chạy được ở notebook bằng `statsmodels.formula.api.ols` + `anova_lm`.

🎯 **Phương pháp này trả lời câu hỏi gì?** ANOVA hai chiều kiểm tra ĐỒNG THỜI 3 câu hỏi: (1) biến A (trường) có ảnh hưởng chính hay không, (2) biến B (giới tính) có ảnh hưởng chính hay không, (3) ẢNH HƯỞNG của A có PHỤ THUỘC vào mức của B hay không (hiệu ứng tương tác). Câu hỏi thứ 3 KHÔNG thể trả lời được bằng cách chạy 2 ANOVA một chiều riêng rẽ — đây chính là lý do cần ANOVA hai chiều.

In [ ]:
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm

model = smf.ols("math ~ C(school) * C(gender)", data=df).fit()
aov = anova_lm(model, typ=2)
print(aov.round(4))

#### 📤 Đầu ra thật
Trường (hiệu ứng chính): F≈5,81, p≈0,003 → có ý nghĩa. Giới tính (hiệu ứng chính): F≈1,71, p≈0,192 → không có ý nghĩa. Trường×Giới tính (tương tác): F≈1,16, p≈0,315 → không có ý nghĩa. ✅ Khớp đúng bài — trường tạo khác biệt thật, nhưng KHÔNG khác nhau đáng kể giữa nam/nữ (không có tương tác).

## 4. Trực quan hoá tương tác (hai đường gần song song = không có tương tác mạnh)

In [ ]:
fig, ax = plt.subplots()
sns.pointplot(data=df, x="school", y="math", hue="gender", order=["A","B","C"], ax=ax, errorbar=None)
ax.set_title("Điểm Toán trung bình theo trường × giới tính")
plt.show()